## NUMPY and Pandas

#### Numpy Internals

In [38]:
# Numpy Metadata
import numpy as np
a = np.arange(6).reshape(2, 3)
print(a)
print(a.shape)
print(a.ndim)
print(a.dtype)
print(a.strides)

[[0 1 2]
 [3 4 5]]
(2, 3)
2
int64
(24, 8)


In [36]:
# Memory Layout
import numpy as np
a = np.array([[1, 2, 3], [4, 5, 6]])
print(a.flags['C_CONTIGUOUS']) # True - row major (default)
print(a.flags['F_CONTIGUOUS']) # False

b = np.asfortranarray(a)
print(b.flags['C_CONTIGUOUS']) # False
print(b.flags['F_CONTIGUOUS']) # True - column major

True
False
False
True


In [39]:
# Dtype Selection
X = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
print(X.dtype) # float64

X = np.array([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]], dtype = np.float32)
print(X.dtype) # float32

X = X.astype(np.float32)
print(X.dtype)

float64
float32
float32


In [41]:
# shape, ndim and strides - Metadata Trio
a = np.zeros((3, 4, 5), dtype = np.float32)
print(a)
print(a.shape)
print(a.ndim)
print(a.strides)

print('=' * 50)

b = a.T
print(b.strides) # strides are reversed, same buffer
print(b.base is a) # True - no copy made

[[[0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]]

 [[0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]
  [0. 0. 0. 0. 0.]]]
(3, 4, 5)
3
(80, 20, 4)
(4, 20, 80)
True


In [45]:
# Views vs Copies
# Views share same memory buffer as the original
a = np.array([1,2,3,4,5,6])
b = a[1:4]
b[0] = 999
print(a)

# Copy has its own independent buffer.
a = np.array([1,2,3,4,5])
b = a[1:4].copy()
b[0] = 1080
print(a)

print('=' * 50)

print(b.base is a) # True if b is a view of a
print(b.base is None) # True if b owns its data

[  1 999   3   4   5   6]
[1 2 3 4 5]
False
True


In [47]:
# np.ascontiguousarray() and np.asfortranarray()
a = np.zeros((3, 4), dtype = np.float32)
b = a.T
print(b.flags['F_CONTIGUOUS'])
c = np.ascontiguousarray(b)
print(c.flags['C_CONTIGUOUS'])
d = np.asfortranarray(c)
print(d.flags['F_CONTIGUOUS'])

True
True
True


In [49]:
# np.frombuffer() = Zero Copy Interop: when data arrives as raw data it creates a numpy array that reads
# directly from those bytes - no copy
raw_bytes = b'\x00\x00\x80\x3f\x00\x00\x00\x40'   # two float32 values: 1.0 and 2.0
a = np.frombuffer(raw_bytes, dtype = np.float32)
print(a) # it is read only - to modify use .copy()

[1. 2.]


In [53]:
# Exercise Questions
# Ex1: Strides Intuition
import numpy as np
a = np.arange(12, dtype = np.float32).reshape(3, 4)
print('Shape: ', a.shape)
print('Strides: ', a.strides)

b = a[:, ::2]
print(f'b strides: {b.strides}')
print(f'b C-CONTIGUOUS: {b.flags["C_CONTIGUOUS"]}')

print('=' * 50)

# Ex2: The view trap
data = np.array([10, 20, 30, 40, 50], dtype = np.float32)
features = data[1:4]
features /= features.max()
print('data: ', data)
print('features: ', features)
print(features.base is data)

print('=' * 50)

# Ex3: Dtype conversion cost
import time
a = np.random.rand(10_000).astype(np.float64)
t1 = time.perf_counter()
b = a.astype(np.float32)
t2 = time.perf_counter()
print(f'Cast time: {(t2 - t1) * 1000:.2f} ms')
print(f'a size: {a.nbytes / 1e6:.1f} MB')
print(f'b size: {b.nbytes / 1e6:.1f} MB')

Shape:  (3, 4)
Strides:  (16, 4)
b strides: (16, 8)
b C-CONTIGUOUS: False
data:  [10.    0.5   0.75  1.   50.  ]
features:  [0.5  0.75 1.  ]
True
Cast time: 0.09 ms
a size: 0.1 MB
b size: 0.0 MB


In [59]:
# Practice Questions
# Q1. You have a 2D array a of shape (1000, 128) with dtype=float64. You want to pass it to a PyTorch 
# model. What is the minimum number of steps to do this correctly, and what goes wrong if you skip them?
a = np.ones((1000, 128), dtype = np.float64)
a = a.astype(np.float32) # min steps = 1, if we omit it then model will throw an error

print('=' * 50)

# Q2. After running b = a.T, is b C-contiguous? What are b's strides if a has shape (3, 4) and
# dtype=float32?
a = np.ones((3, 4), dtype = np.float32)
b = a.T
print(b.flags['C_CONTIGUOUS']) # False, it is F_CONTIGUOUS now
print(b.strides) # row: 4*4 and col: 1*4 i.e. strides = (16, 4) for a and for b it will be reversed (4, 16)

print('=' * 50)

# Q3. What is the difference in memory usage between a = np.ones((1000, 1000), dtype=np.float64) 
# and a = np.ones((1000, 1000), dtype=np.float32)? Why does this matter when training a model 
# on a GPU with 8 GB VRAM?
a = np.ones((1000, 1000), dtype = np.float64)
b = np.ones((1000, 1000), dtype = np.float32)
print(f"{a.nbytes / 1e-6:.1f}")
print(f"{b.nbytes / 1e-6:.1f}") # the memory gets half in float32 dn it is important because for large 
# numbers the memory usage is exponential

print('=' * 50)

# Q4. You write labels = dataset['label'][0:100] and then labels[0] = -1. Has dataset['label'] been 
# mutated? How would you check? How would you prevent it?
dataset = {'label' : np.arange(200)}
labels = dataset['label'][0:100]
labels[0] = -1
print(dataset['label']) # Yes it is mutated. To prevent it we can use .copy() here.
print(labels.base is dataset['label'])
print(labels.base is None)

print('=' * 50)

# Q5. In which of the following cases does NumPy return a copy rather than a view?
# (a) a[0:5] (b) a[[0, 1, 2]] (c) a.reshape(5, -1) (C-contiguous) (d) a[a > 0] (e) a.T
a = np.arange(10)
print(a) # here b and d will give copy instead of view. b because we are using list inside list
# also called fancy indexing and d because of boolean indexing

print('=' * 50)

# Hots Question: What are b.shape, b.strides, and is b C-contiguous?
a = np.arange(6, dtype = np.float32).reshape(2, 3)
b = a.T
print(b.shape)
print(b.strides)
print(b.flags['C_CONTIGUOUS'])

False
(4, 16)
8000000000000.0
4000000000000.0
[ -1   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161
 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179
 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197
 198 199]
True
False
[0 1 2 3 4 5 6 7 8 9]
(3, 2)
(4, 12)
False


#### Vectorization and Broadcasting

In [1]:
# Think in Shapes, Not Loops
import time
import numpy as np

# Loop Mindset - Wrong
data = np.zeros((2, 4, 3), dtype = np.float32)
result = []

t1 = time.perf_counter()
for row in data:
    result.append(row - row.mean())
result = np.array(result)
t2 = time.perf_counter()
print(result)
print(f'Time Taken by Loop: {(t2 - t1) * 1000:.2f} ms')

print('-' * 50)

# Shape Mindset - Right
t3 = time.perf_counter()
result = data - data.mean(axis = 1, keepdims = True)
t4 = time.perf_counter()
print(result)
print(f'Time Taken by Loop: {(t4 - t3) * 1000:.2f} ms')

[[[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]]

 [[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]]]
Time Taken by Loop: 0.23 ms
--------------------------------------------------
[[[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]]

 [[0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]
  [0. 0. 0.]]]
Time Taken by Loop: 0.11 ms


In [2]:
print(data.mean(axis = 1, keepdims = True))
print((data.mean(axis = 1, keepdims = True)).shape)

[[[0. 0. 0.]]

 [[0. 0. 0.]]]
(2, 1, 3)


In [3]:
# Vectorization - expressing an operation over entire arrays insted elements
import numpy as np
import time
n = 10_000
a = np.random.rand(n).astype(np.float32)

# Python Loop
t0 = time.perf_counter()
result_loop = [x * 2.0 for x in a]
t1 = time.perf_counter()
print(f'Python Loop: {(t1 - t0) * 1000:.1f} ms')

# Vectorized
t0 = time.perf_counter()
result_vec = a * 2.0
t1 = time.perf_counter()
print(f'Vectorized: {(t1 - t0) * 1000:.1f} ms')

Python Loop: 1.4 ms
Vectorized: 0.1 ms


In [4]:
# Broadcasting Rules: for performing arithemetic b/w arrays of different shapes without copying data
# 1. Scalar broadcast
a = np.array([1,2,3,4,5])
s = np.array(1)
print(a.shape)
print(s.shape)
print(a * s)
print((a * s).shape)

print('-' * 40)

# 2. Row Vector subtracted from matrix
x = np.random.rand(100, 4).astype(np.float32)
mean = x.mean(axis = 0) # shape (4,)
x_centered = x - mean 
print(x_centered.shape)

print('-' * 40)

# 3. Column vector broadcast
row_sums = x.sum(axis = 1, keepdims = True)
x_norm = x / row_sums
print(x_norm.shape)

print('-' * 40)

# 4. Two vectors --> outer product
a = np.random.rand(3, 1)
print(a)
print(a.shape)
b = np.random.rand(1, 4)
print(b)
print(b.shape)
outer = a * b
print(outer)
print(outer.shape)

(5,)
()
[1 2 3 4 5]
(5,)
----------------------------------------
(100, 4)
----------------------------------------
(100, 4)
----------------------------------------
[[0.24490537]
 [0.94175214]
 [0.16730968]]
(3, 1)
[[0.3262516  0.51308067 0.24035023 0.72573158]]
(1, 4)
[[0.07990077 0.12565621 0.05886306 0.17773556]
 [0.30724815 0.48319483 0.22635035 0.68345928]
 [0.05458505 0.08584337 0.04021292 0.12142192]]
(3, 4)


In [5]:
# keepdims = True
x = np.random.rand(5, 3).astype(np.float32)
means = x.mean(axis = 1)
# print(x - means) # Throws error because the dim is dropped and broadcasting breaks

means = x.mean(axis = 1, keepdims = True)
print(x - means)
print((x - means).shape)

[[ 0.38985145 -0.31852958 -0.07132196]
 [-0.01180083 -0.19699895  0.20879984]
 [ 0.43079478 -0.21951139 -0.21128345]
 [ 0.00608931  0.06492926 -0.07101859]
 [-0.01925188 -0.4121344   0.4313864 ]]
(5, 3)


In [8]:
# np.where(condition, x, y) = select element wise from x where condition is True and y where False
scores = np.array([0.2, 0.7, 0.4, 0.9, 0.1])
print(scores)

labels = np.where(scores > 0.5, 1, 0)  # if scores > 0.5 then output is 1 else 0
print(labels)
clipped = np.where(scores > 0.5, scores, 0.0)
print(clipped)

# np.where(condition) with no x/y returns indices where condition is True (equivalent to np.argwhere)
indices = np.where(scores > 0.5)[0]
print(indices)

[0.2 0.7 0.4 0.9 0.1]
[0 1 0 1 0]
[0.  0.7 0.  0.9 0. ]
[1 3]


In [11]:
# np.select() - when we need more than one condition, np.select() avoids nested np.where()
scores = np.array([0.1, 0.45, 0.6, 0.85, 0.95])
conditions = [
    scores < 0.3, scores < 0.6, scores < 0.9, scores >= 0.9
]
choices = ['low', 'medium', 'high', 'very high']
rating = np.select(conditions, choices, default = 'unknown')
print(rating)
print(rating.dtype)

['low' 'medium' 'high' 'high' 'very high']
<U9


In [15]:
# Universal Functions (ufuncs) - every numpy operation (np.add, np.multiply, np.sqrt, np.exp, np.log, etc)
# is a ufunc - a compiled function that: operates element eise over array, supports broadcasting 
# automatically and can operate in-place (with out = parmeter)
a = np.array([1.0, 4.0, 9.0], dtype = np.float32)
print(np.sqrt(a))
print(np.sqrt(a, out = a))

print('=' * 40)

# Creating custom ufunc = Use np.frompyfunc() or np.vectorize() to wrap a python fn into ufunc-like object
def clip_and_log(x):
    return np.log(max(x, 1e-7))
vfunc = np.vectorize(clip_and_log)
result = vfunc(np.array([-0.1, 0.0, 0.5, 2.0]))
print(result)

[1. 2. 3.]
[1. 2. 3.]
[-16.11809565 -16.11809565  -0.69314718   0.69314718]


In [18]:
# Speed Comparison
import numpy as np
import time

n = 5_000
a = np.random.rand(n).astype(np.float32)
b = np.random.rand(n).astype(np.float32)

def time_it(label, fn):
    t0 = time.perf_counter()
    result = fn()
    t1 = time.perf_counter()
    print(f'{label:<30} {(t1 - t0) * 1000:8.2f} ms')
    return result

# Python Loop
time_it('Python loop', lambda: [a[i] + b[i] for i in range(n)])

# np.vectorize
vf = np.vectorize(lambda x, y: x + y)
time_it('np.vectorize', lambda: vf(a, b))

# Native vectorized
time_it('Vectorized (a + b)', lambda: a + b)

Python loop                        1.31 ms
np.vectorize                       0.70 ms
Vectorized (a + b)                 0.08 ms


array([1.7026297 , 1.2282244 , 0.7969985 , ..., 1.2042372 , 1.1763953 ,
       0.63768524], shape=(5000,), dtype=float32)

In [23]:
# Common Broadcastimg Bugs
# 1. Ambiguous Subtraction without keepdims
X = np.ones((4, 3))
col_means = X.mean(axis=0)      # shape (3,)  ← fine, broadcasts as (1, 3)
# row_means = X.mean(axis=1)      # shape (4,)  ← DANGER: broadcasts as (4,) not (4,1)
row_means = X.mean(axis=1, keepdims=True) # this is right
print(X - row_means)

print('-' * 40)

# 2. Unintended singleton dim creating a huge result
a = np.ones(5)
b = np.ones(5)
a = a.reshape(5, 1) # accidental reshape, we wanted shape (5,)
print(a * b)
a = a.reshape(1, 5)
print(a * b)

print('-' * 40)

# 3. Integer arithmetic silently losing precision
a = np.array([1, 2, 3]) # int64 dtype by default
result = a / 2
print(result) # gives float output
result = a // 2
print(result) # gives int floor division, no error


[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
----------------------------------------
[[1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]
 [1. 1. 1. 1. 1.]]
[[1. 1. 1. 1. 1.]]
----------------------------------------
[0.5 1.  1.5]
[0 1 1]


In [29]:
# Exercise Questions
# 1. Trace the braodcast
import numpy as np
a = np.arange(12).reshape(3, 4).astype(np.float32)
v = np.array([1, 2, 3, 4], dtype = np.float32)
result = a * v
print(result.shape)
print(result)

print('-' * 40)

# 2. Row wise softmax
X = np.random.rand(5, 4).astype(np.float32)
exp_X = np.exp(X - X.max(axis=1, keepdims=True))   # subtract max for numerical stability
softmax = exp_X / exp_X.sum(axis=1, keepdims=True)
print(softmax.sum(axis=1))   # [1. 1. 1. 1. 1.]

print('-' * 40)

# 3. Vectorized distance matrix
points = np.random.rand(6, 2).astype(np.float32)
sq_norms = np.sum(points**2, axis=1, keepdims=True)
dist_squared = sq_norms + sq_norms.T - 2 * (points @ points.T)
dists = np.sqrt(np.maximum(dist_squared, 0))
print("Final Distance Matrix Shape:", dists.shape) # (6, 6)
print("\nDistance Matrix:\n", dists)

print('-' * 40)

# 4. np.where in loss fn
y_true = np.array([1.0, 2.0, 3.0, 4.0, 5.0], dtype=np.float32)
y_pred = np.array([1.1, 2.5, 0.5, 4.0, 9.0], dtype=np.float32)
delta = 1.0
residuals = np.abs(y_pred - y_true)
loss = np.where(
    residuals <= delta, 
    0.5 * (residuals ** 2), 
    delta * (residuals - 0.5 * delta)
)
print("Loss per element:", loss)
print("Mean Huber Loss:", loss.mean())

(3, 4)
[[ 0.  2.  6. 12.]
 [ 4. 10. 18. 28.]
 [ 8. 18. 30. 44.]]
----------------------------------------
[1.         0.99999994 1.         0.9999999  0.99999994]
----------------------------------------
Final Distance Matrix Shape: (6, 6)

Distance Matrix:
 [[0.         0.38992155 0.6589501  0.8006203  0.31104338 0.5481846 ]
 [0.38992155 0.         1.0025294  0.96653956 0.59664613 0.26367232]
 [0.6589501  1.0025294  0.         0.5340412  0.41259766 1.0374668 ]
 [0.8006203  0.96653956 0.5340412  0.         0.50275826 0.8599797 ]
 [0.31104338 0.59664613 0.41259766 0.50275826 0.         0.62962055]
 [0.5481846  0.26367232 1.0374668  0.8599797  0.62962055 0.        ]]
----------------------------------------
Loss per element: [0.005 0.125 2.    0.    3.5  ]
Mean Huber Loss: 1.126


In [54]:
# Practice Questions
# Q1. What shape does NumPy produce for np.ones((3, 1, 4)) + np.ones((5, 4))? Walk through the 
# broadcasting rules step by step.
a = np.ones((3, 1, 4))
b = np.ones((5, 4))
print(a + b)
print((a + b).shape)

print('=' * 60)

# Q2. You have a prediction array probs of shape (1000, 10) (1000 samples, 10 classes). You want 
# to divide each row by its maximum value. Write the single vectorized expression. What would 
# go wrong if you forgot keepdims=True?
probs = np.random.rand(1000, 10).astype(np.float32)
m = np.max(probs, axis = 1, keepdims = True)
print(probs.shape)
print(m.shape)
print((probs / m).shape)

print('=' * 60)

# Q3. What is the difference in behavior between np.where(a > 0) and np.where(a > 0, a, 0.0)? 
# Give the output shape for each when a has shape (4, 3).
a = np.random.randn(4, 3)
m = np.where(a > 0)[0] # here since only condition is given and x/y is missing it will return the 
# indices of elements that match the condition
n = np.where(a > 0, a, 0.0) # here the output will be element itself if it is > 0 else 0
print(m.shape)
print(n.shape)

print('=' * 60)

# Q4. Why is np.vectorize not a performance tool? In what situation would you legitimately reach for it?
x = np.array([5.0, -2.0, 3.0, -7.0])
def clip_element(element):
    return max(element, 0.0)
v = np.vectorize(clip_element)
result = v(x)
print(result)

print('=' * 60)

# Q5. Write a single vectorized expression (no loops) to compute the z-score normalization of a matrix 
# X of shape (N, D) — subtract the per-feature mean and divide by the per-feature standard deviation.
a = np.ones((2, 3))
mean = a.mean(axis = 0, keepdims = True)
std = a.std(axis = 0, keepdims = True)
z = (a - mean) / (std + 1e-8)
print(z)

[[[2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]]

 [[2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]]

 [[2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]
  [2. 2. 2. 2.]]]
(3, 5, 4)
(1000, 10)
(1000, 1)
(1000, 10)
(8,)
(4, 3)
[5. 0. 3. 0.]
[[0. 0. 0.]
 [0. 0. 0.]]


#### Indexing, Slicing and Advanced Fancy Indexing

In [57]:
# Basic Slicing 
import numpy as np

# They returns views and not copies
a = np.arange(10)
print(a[2 : 7])
print(a[: : 2])
print(a[: : -1])
print(a[-3 : ])
print(a[: -3])

b = a[2: 5]
b[:] = 99
print(a)

# To get copy we use .copy()
c = a[2:5].copy()
c[:] = 34
print(a)

[2 3 4 5 6]
[0 2 4 6 8]
[9 8 7 6 5 4 3 2 1 0]
[7 8 9]
[0 1 2 3 4 5 6]
[ 0  1 99 99 99  5  6  7  8  9]
[ 0  1 99 99 99  5  6  7  8  9]


In [62]:
# Multi Dimensional Slicing
x = np.arange(20).reshape(4, 5)
print(x)
print(x[1, 3])
print(x[1, :])
print(x[:, 2])
print(x[:, 2 : 4])
print(x[1 : 3, 1 : 4])
print(x[: : 2, : : 2])

print('=' * 40)

# Shape collapse trap: X[:, 2] gives shape (4,) and not (4, 1). To keep the dim for broadcasting we use:
print(x[:, 2 : 3].shape) # Preserves axis
print(x[:, [2]].shape) # fancy indexing, also works (but copies!)
print(x[:, 2, None].shape) # adds axis after selection... but x is 2D, so:
print(x[:, 2][:, None].shape)

[[ 0  1  2  3  4]
 [ 5  6  7  8  9]
 [10 11 12 13 14]
 [15 16 17 18 19]]
8
[5 6 7 8 9]
[ 2  7 12 17]
[[ 2  3]
 [ 7  8]
 [12 13]
 [17 18]]
[[ 6  7  8]
 [11 12 13]]
[[ 0  2  4]
 [10 12 14]]
(4, 1)
(4, 1)
(4, 1)
(4, 1)


In [78]:
# Fancy Indexing - Selecting with an array of indices: always returns a copy
a = np.array([10, 20, 30, 40, 50])
idx = np.array([0, 2, 4])
print(a[idx]) # copy
print(a[[0, 2, 4]]) # same - list literal also works

print('=' * 40)

# Fancy Indexing in 2D: Row and Column Selection
x = np.arange(20).reshape(4, 5)
rows = np.array([0, 2, 3])
print(x[rows])

cols = np.array([1, 3])
print(x[:, cols])

print(x[rows][:, cols])

print('=' * 40)

# Fancy Indexing vs Cross-Product
x = np.arange(7, 65, 3).reshape(4, 5)
print(x)
print(x[[0, 2, 3], [1, 3, 0]]) # paired: selects (0, 1), (2,3), (3, 0)  - three scalars
print(x[np.ix_([0, 2, 3], [1, 3])]) # cross product: selects all combos of rows and cols

[10 30 50]
[10 30 50]
[[ 0  1  2  3  4]
 [10 11 12 13 14]
 [15 16 17 18 19]]
[[ 1  3]
 [ 6  8]
 [11 13]
 [16 18]]
[[ 1  3]
 [11 13]
 [16 18]]
[[ 7 10 13 16 19]
 [22 25 28 31 34]
 [37 40 43 46 49]
 [52 55 58 61 64]]
[10 46 52]
[[10 16]
 [40 46]
 [55 61]]


In [79]:
# np.ix_ = Multi - Dim cross product selection: takes 1D index arrays and returns a tuple of reshaped 
# index arrays that broadcast all combos
x = np.arange(7, 65, 3).reshape(4, 5)
print(x)
rows = [0, 2]
cols = [1, 3, 4]
print(x[np.ix_(rows, cols)])

print('=' * 40)

logits = np.random.rand(5, 10).astype(np.float32)
true_labels = np.array([2, 7, 0, 4, 9])
correct_logits = logits[np.arange(5), true_labels]
print(correct_logits)

[[ 7 10 13 16 19]
 [22 25 28 31 34]
 [37 40 43 46 49]
 [52 55 58 61 64]]
[[10 16 19]
 [40 46 49]]
[0.647776   0.8486285  0.65347695 0.7658166  0.811026  ]


In [94]:
# Boolean Masking - it is an array of True/False values the same shape as array being indexed.
# It selects elements where mask is True. Boolean masking always returns a copy
a = np.array([3, -1, 4, -1, 5, -9, 2, 6])
mask = a > 0 # gives true, false values
print(a[mask])
print(a > 0) # same, inline

a[a < 0] = 0 # modifying in-place though a mask
print(a)

print('=' * 40)

# 2D Boolean Masking
x = np.random.randn(4, 3).astype(np.float32)
mask = x > 0 # dtype bool
row_mask = (x > 0).all(axis = 1) # shape (4,) dtype bool
print(x[row_mask])
print(x[row_mask].shape)

print('=' * 40)

# Combining Conditios
a = np.array([1, 5, 3, 8, 2, 9, 4, 7])
mask = (a >= 3) & (a <= 7) # and
print(a[mask])

mask = (a < 2) | (a > 8) # or
print(a[mask])

mask = ~(a > 5) # bitwise -- NOT
print(a[mask])

[3 4 5 2 6]
[ True False  True False  True False  True  True]
[3 0 4 0 5 0 2 6]
[[1.1509404  1.4081255  0.59453183]]
(1, 3)
[5 3 4 7]
[1 9]
[1 5 3 2 4]


In [101]:
# np.argsort(), np.argmax(), np.argmin() - return indices rather than values
# np.argmin() and np.argmax()
scores = np.array([0.1, 0.6, 0.2, 0.05, 0.05])
print(np.argmin(scores))
print(np.argmax(scores))

print('-' * 40)

# 2D: argmax per row (predicted class for each sample)
logits = np.random.rand(5, 10).astype(np.float32)
predicted_classes = np.argmax(logits, axis = 1)
print(predicted_classes)

print('-' * 40)

# np.argsort()
order = np.argsort(scores)
print(order)
print(scores[order])

# argsort() in descending order
order_desc = np.argsort(scores)[::-1]
print(scores[order_desc])

print('-' * 40)

# Top-K pattern 
scores = np.array([0.3, 0.1, 0.8, 0.5, 0.2, 0.9, 0.4])
k = 3
top_k_indices = np.argsort(scores)[::-1][:k]
print(top_k_indices)
top_k_scores = scores[top_k_indices]
print(top_k_scores)

print('-' * 40)


# np.argpartition() - faster for large arrays, result is NOT sorted within the k
top_k_idx = np.argpartition(scores, -k)[-k :] # top-k indices, unordered
print(top_k_idx)
top_k_idx_sorted = top_k_idx[np.argsort(scores[top_k_idx])[::-1]] # sort those k
print(top_k_idx_sorted)

3
1
----------------------------------------
[1 1 4 8 9]
----------------------------------------
[3 4 0 2 1]
[0.05 0.05 0.1  0.2  0.6 ]
[0.6  0.2  0.1  0.05 0.05]
----------------------------------------
[5 2 3]
[0.9 0.8 0.5]
----------------------------------------
[3 2 5]
[5 2 3]


In [105]:
# np.take() and np.put()
# np.take(a, indices, axis) is fancy indexing with explicit axis control.
x = np.arange(20).reshape(4, 5)
indices = [0, 2]
np.take(x, indices, axis = 0) # select rows 0 and 2 - shape (2, 5)
np.take(x, indices, axis = 1) # select cols 0 and 2 - shape (4, 5)
print(x[indices])
print(x[:, indices]) # we can directly print the np.take statements as well for same outputs
# In embeddings loopup - np.take(emb_matrix, token_ids, axis = 0) gathers embeddings for seq of token IDs

print('=' * 40)

# np.put(a, indices, values) writes values to specific flat indices. Operates on the flattened array
a = np.zeros(10)
indices = [2, 5, 8]
values = [1.0, 2.0, 3.0]
np.put(a, indices, values) # inserted the values in given indices in array a.
print(a)

[[ 0  1  2  3  4]
 [10 11 12 13 14]]
[[ 0  2]
 [ 5  7]
 [10 12]
 [15 17]]
[0. 0. 1. 0. 0. 2. 0. 0. 3. 0.]


In [111]:
# Structured Arrays
dtype = np.dtype([('name', 'U20'), ('age', np.int32), ('score', np.float32)])
data = np.array([('Alice', 30, 0.92), ('Bob', 25, 0.85)], dtype = dtype)

print(data['name'])
print(data['score'])
print(data[0])

['Alice' 'Bob']
[0.92 0.85]
('Alice', 30, 0.92)


In [118]:
# Complete Practical Pattern 
import numpy as np
np.random.seed(42)
logits = np.random.randn(8, 5).astype(np.float32)
pred_classes = np.argmax(logits, axis = 1)
print(pred_classes.shape)

exp_l = np.exp(logits - logits.max(axis = 1, keepdims = True))
probs = exp_l / exp_l.sum(axis = 1, keepdims = True)

confidence = probs[np.arange(8), pred_classes]

high_conf_mask = confidence > 0.4
reliable_preds = pred_classes[high_conf_mask]

top3_indices = np.argsort(probs, axis = 1)[:, ::-1][:, :3]
print('Predicted Classes: ', pred_classes)
print('Confidence: ', confidence.round(3))
print('High-conf preds: ', reliable_preds)
print('Top-3 per sample: \n', top3_indices)

(8,)
Predicted Classes:  [3 1 2 2 0 2 1 1]
Confidence:  [0.468 0.478 0.446 0.464 0.617 0.348 0.605 0.391]
High-conf preds:  [3 1 2 2 0 1]
Top-3 per sample: 
 [[3 2 0]
 [1 2 4]
 [2 0 1]
 [2 0 3]
 [0 2 1]
 [2 0 4]
 [1 4 2]
 [1 4 0]]


In [48]:
# Exercise Questions
# 1. Fancy Indexing vs Slicing = Predict the shape and whether it's a view or copy for each:
import numpy as np
x = np.arange(25).reshape(5, 5)
a = x[1:3, 2:4]
b = x[[1, 3], :] 
c = x[np.ix_([0,2,4], [1,3])]
d = x[x > 10] 

print(a.shape, a.base is x)   # shape (2, 2), slicing, view
print(b.shape, b.base is x)   # shape (2, 5), fancy indexing, copy
print(c.shape, c.base is x)   # shape (3, 2), fancy indexing, copy
print(d.shape, d.base is x)   # shape (14, ), boolean indexing, copy

print('=' * 40)

# 2. The argmax pipeline = Given multi-class probabilities, compute accuracy without any loop
y_true = np.array([2, 0, 1, 3, 2, 1, 0, 3])
logits = np.random.rand(8, 4).astype(np.float32)
y_pred = np.argmax(logits, axis = 1)
accuracy = np.mean(y_true == y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')

print('=' * 40)


# 3. Boolean masking on dataset = Select scores where age > 40 AND score > 0.5
np.random.seed(0)
ages = np.random.randint(18, 70, size = 10)
scores = np.random.rand(10)
indices = (ages > 40) & (scores > 0.5)
print(ages[indices], scores[indices])

print('=' * 40)

# 4. Top-k retrieval = Return the indices of the top-5 documents, sorted by score descending
# Use np.argpartition for O(n) efficiency, then sort the k results
scores = np.random.rand(1000).astype(np.float32)
k = 5
topk_indices = np.argpartition(scores, -k)[-k:]
final_topk_values = topk_indices[np.argsort(scores[topk_indices])[::-1]]
print('Top 5 Indices: ', final_topk_values)
print('Top 5 Values: ', scores[final_topk_values])

print('=' * 40)

(2, 2) False
(2, 5) False
(3, 2) False
(14,) False
Accuracy: 25.00%
[65 57] [0.891773   0.52889492]
Top 5 Indices:  [958 603 133 579 462]
Top 5 Values:  [0.99980855 0.999278   0.998847   0.99796224 0.9944008 ]


In [94]:
# Practice Qns
# Q1. What is the shape of X[np.ix_([0, 2], [1, 3, 4])] if X has shape (5, 6)? Is it a view or a copy?
X = np.random.rand(5, 6)
out = X[np.ix_([0, 2], [1, 3, 4])]
print(out.shape) # shape should be (2, 3)
print(out.base is X) # Fancy Indexing, hence copy

print('=' * 40)

# Q2. You have probs of shape (32, 100) (batch of 32, 100 classes) and labels of shape (32,). Write 
# a single expression to extract the predicted probability for the true class of each sample.
batch = 32
classes = 100
probs = np.random.rand(batch, classes).astype(np.float32)
labels = np.random.randint(0, classes, size=(batch,))
true_class_probs = probs[np.arange(batch), labels]
print("Output shape:", true_class_probs.shape)

print('=' * 40)

# Q3. What does ~mask do when mask is a boolean array? Give an example where you'd use it in an ML 
# pipeline.
a = np.array([1, 3, 5, 6, 2, 8, 7, 9])
mask = ~(a >= 5)
print(a[mask])
# ~mask here ~ is a bitwise NOT operator and returns False values as True and vice versa

print('=' * 40)

# Q4. You need to select rows from a matrix X where the value in column 3 is greater than 0.5. Write 
# the boolean masking expression. Does this return a view or a copy?
matrix = np.random.rand(7, 5).astype(np.float32)
print(matrix, '\n\n')
masking = matrix[matrix[:, 3] > 0.5]
print(masking)
print(matrix.base is masking) # it will give a copy

print('=' * 40)

# Q5. What is the difference between np.argmax(a) and np.argsort(a)[-1]? When would each be preferred?
a = np.array([5, 7, 9, 2, 1, 6, 3, 8])
print(np.argmax(a)) # returns the index values or highest value in a 
print(np.argsort(a)[-1]) # sort the values of a in asc order and then return the -1 index values original 
# index number

print('=' * 40)

# Hots Question: logits[np.arange(N), labels] ---> why np.arange(N) is needed, and why logits[:, labels] 
# would give the wrong result
N = 5
logits = np.random.randn(5, 4).astype(np.float32)
labels = np.random.randint(0, 4, size=(5,))
correct = logits[np.arange(N), labels]
print("Shape:", correct.shape)  # Output: (5,)
print(correct)

print("\n--- The Wrong Way ---")
wrong = logits[:, labels]
print("Shape:", wrong.shape)    # Output: (5, 5) — Unwanted 2D Grid!
print(wrong)

(2, 3)
False
Output shape: (32,)
[1 3 2]
[[0.7700017  0.5523559  0.8381402  0.41220975 0.73995435]
 [0.43112218 0.7316244  0.07300718 0.11271311 0.5008624 ]
 [0.70618856 0.19462793 0.6988929  0.49389344 0.964414  ]
 [0.4042049  0.52473366 0.8225078  0.4246611  0.1847822 ]
 [0.80963755 0.10193383 0.83639854 0.5734477  0.3626722 ]
 [0.71799546 0.48279077 0.12710565 0.826115   0.64183056]
 [0.19060782 0.65225065 0.591611   0.93594134 0.75418645]] 


[[0.80963755 0.10193383 0.83639854 0.5734477  0.3626722 ]
 [0.71799546 0.48279077 0.12710565 0.826115   0.64183056]
 [0.19060782 0.65225065 0.591611   0.93594134 0.75418645]]
False
2
2
Shape: (5,)
[-1.4687599   0.34515196 -0.7867555  -0.32371482 -0.41962835]

--- The Wrong Way ---
Shape: (5, 5)
[[-1.4687599  -1.2653775  -1.4687599   0.84355545 -0.3856716 ]
 [-0.6321769   0.34515196 -0.6321769   1.6142037  -0.6110151 ]
 [-0.7867555   0.75631493 -0.7867555   0.530222   -0.830534  ]
 [-0.8713518   0.20344482 -0.8713518  -0.32371482 -0.27953458]
 

#### Linear Algebra Operations

In [99]:
# np.dot() vs @ vs np.matmul() = all three used for matrix multiplicaton
# For 2D arrays (matrices) - all three could be used
import numpy as np
a = np.random.rand(3, 4).astype(np.float32)
b = np.random.rand(4, 5).astype(np.float32)
c1 = np.dot(a, b)
c2 = a @ b
c3 = np.matmul(a, b)
print(c1.shape)
print(c2.shape)
print(c3.shape)
print(np.allclose(c1, c2) and np.allclose(c2, c3))

print('=' * 50)

# For 1D vectors - dot product only
a = np.array([1., 2., 3.])
b = np.array([4., 5., 6.])
print(np.dot(a, b))
print(np.matmul(a, b))
print(a @ b)

print('=' * 50)

# For 3D+ (batched) arrays - @ and np.matmul are used
a = np.random.rand(2, 3, 4).astype(np.float32)
b = np.random.rand(2, 4, 5).astype(np.float32)
print(np.matmul(a, b).shape)
print((a @ b).shape)

(3, 5)
(3, 5)
(3, 5)
True
32.0
32.0
32.0
(2, 3, 5)
(2, 3, 5)


In [119]:
# np.linalg
# np.linalg.norm() - computes distances
v = np.array([3., 4.])
print(np.linalg.norm(v)) # L2 norm (euclidean length)
print(np.linalg.norm(v, ord = 1)) # L1 norm (Manhattan)
print(np.linalg.norm(v, ord = np.inf)) # Linf norm (max absolute value)

x = np.random.rand(5, 128).astype(np.float32)
row_norms = np.linalg.norm(x, axis = 1, keepdims = True)
print(row_norms.shape)
x_normalized = x / row_norms
print(x_normalized.shape)

print('=' * 40)

# np.linalg.inv() - Matrix inverse
a = np.array([[2., 1.], [1., 3.]]).astype(np.float32)
a_inv = np.linalg.inv(a) # avoid using np.linalg.inv() it is unstable instead use nplinalg.solve(a, b)
print(a)
print(a_inv)
print(a @ a_inv)

print('--' * 20)

x = np.random.rand(100, 5).astype(np.float32)
y = np.random.rand(100).astype(np.float32)
# Bad: w = np.linalg.inv(X.T @ X) @ X.T @ y
w = np.linalg.solve(x.T @ x, x.T @ y)
print(w)

print('=' * 40)

# np.linalg.eig() and np.linalg.svd() - Decompositions
# Eigendecomposition — symmetric matrices (covariance matrices)
X = np.random.rand(100, 5).astype(np.float32)
y = np.random.rand(100).astype(np.float32)

cov = np.cov(X.T)                      # covariance matrix, shape (5, 5)
print(cov.shape)
eigenvalues, eigenvectors = np.linalg.eigh(cov)   # eigh for symmetric
print(eigenvalues)
print(eigenvectors)

print('--' * 20)

# SVD — the workhorse of PCA, matrix factorization, truncated decompositions
U, S, Vt = np.linalg.svd(X, full_matrices=False)
print(U.shape)
print(S.shape)
print(Vt.shape)
# X ≈ U @ np.diag(S) @ Vt
# U: (100, 5), S: (5,), Vt: (5, 5)

print('--' * 20)

# PCA in 3 lines via SVD:
X_centered = X - X.mean(axis=0)
print(X_centered.shape)
U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)
print(U.shape)
print(S.shape)
print(Vt.shape)
X_pca_3d = X_centered @ Vt[:3].T      # project to 3 principal components
print(X_pca_3d.shape)

5.0
7.0
4.0
(5, 1)
(5, 128)
[[2. 1.]
 [1. 3.]]
[[ 0.6 -0.2]
 [-0.2  0.4]]
[[1.0000000e+00 0.0000000e+00]
 [1.4901161e-08 1.0000000e+00]]
----------------------------------------
[0.3222287  0.12614423 0.14111882 0.20018631 0.11063702]
(5, 5)
[0.06803359 0.08079281 0.09008682 0.09261644 0.11019151]
[[ 0.75731132  0.13631142 -0.47118418  0.43061486 -0.02133237]
 [-0.22377827  0.30598403 -0.63190179 -0.42094568 -0.52896306]
 [-0.02992499  0.69113564 -0.0800717  -0.21992512  0.68312333]
 [-0.06123163  0.63908967  0.46416915  0.39003348 -0.46929305]
 [-0.60971964 -0.04109613 -0.39600645  0.6609708   0.18124447]]
----------------------------------------
(100, 5)
(5,)
(5, 5)
----------------------------------------
(100, 5)
(100, 5)
(5,)
(5, 5)
(100, 3)


In [128]:
# np.einsum() - for expressing tensor operations using index notation.
# np.einsum('subscript_string', *operands) each letter in subscript represents an axis. Axis that appear
# in input but not in output are summed over (contracted)

# Building up from scratch
# Level 1 - Element wise multiply (no summation)
a = np.array([1., 2., 3.])
b = np.array([4., 5., 6.])
print(np.einsum('i, i->i', a, b))

# Level 2 - Dot Product (sum over the shared axis)
print(np.einsum('i,i->', a, b))

# Level 3 - Matrix vector multiply
a = np.random.rand(3, 4).astype(np.float32)
v = np.random.rand(4).astype(np.float32)
print(np.einsum('ij,j->i', a, v)) # Equivalent: a @ v

# Level 4 - Matrix multiply
a = np.random.rand(3, 4).astype(np.float32)
b = np.random.rand(4, 5).astype(np.float32)
print(np.einsum('ij,jk->ik', a, b)) # Equivalent: a @ b

# Level 5 - Outer product (no shared axes -> no summation)
a = np.array([1., 2., 3.])
b = np.array([4., 5., 6.])
print(np.einsum('i,j->ij', a, b)) # Equivalent: a[:, None] * b[None, :]

# Level 6 - Trace (sum of diagnal)
a = np.random.rand(4, 4).astype(np.float32)
print(np.einsum('ii->', a)) # Equivalent: np.trace(A)

# Level 7 - Transpose
a = np.random.rand(3, 4).astype(np.float32)
print(np.einsum('ij->ji', a))

# Level 8 -Batch matrix multiply
a = np.random.rand(2, 5, 8).astype(np.float32)
b = np.random.rand(2, 8, 6).astype(np.float32)
print(np.einsum('bij,bjk->bik', a, b)) # Equivalent: a @ b   (same for matmul/@)

# Level 9 - Attention Scores
# Q: queries  (batch, heads, seq_len, d_k)
# K: keys     (batch, heads, seq_len, d_k)
# Attention score = Q @ K.T / sqrt(d_k)

batch, heads, seq_len, d_k = 2, 4, 10, 32
Q = np.random.rand(batch, heads, seq_len, d_k).astype(np.float32)
K = np.random.rand(batch, heads, seq_len, d_k).astype(np.float32)

# einsum: for each batch b, head h, query position i, key position j
#         sum over d_k dimension (d)
scores = np.einsum('bhid,bhjd->bhij', Q, K)   # shape (batch, heads, seq_len, seq_len)
scores /= np.sqrt(d_k)

[ 4. 10. 18.]
32.0
[1.7967882 0.5285662 1.0424199]
[[0.3615687  1.4537985  1.2343804  0.38986665 1.3994783 ]
 [0.90312743 1.9157307  1.4893895  0.52047485 1.9823576 ]
 [1.2168427  1.642647   1.3233407  0.63734806 1.9378486 ]]
[[ 4.  5.  6.]
 [ 8. 10. 12.]
 [12. 15. 18.]]
2.3376858
[[0.53851163 0.66306007 0.08525983]
 [0.33296293 0.32767305 0.91303533]
 [0.21538003 0.2511487  0.6403293 ]
 [0.72341025 0.63660175 0.46517202]]
[[[3.662155  3.1196558 3.0898304 2.5285435 2.381714  2.7877262]
  [3.3082368 2.121256  2.4073133 2.1075823 2.1193628 2.8120453]
  [2.3212054 1.6817067 2.0876262 1.9257059 1.4292327 1.4009446]
  [2.5515661 1.5887829 1.6711639 1.86597   1.475023  2.2535908]
  [2.982424  2.2088425 2.2025397 2.4274938 1.8539559 2.6742513]]

 [[2.2044408 1.7819818 2.1294227 1.3133043 1.9880673 1.7776399]
  [2.1858683 1.3477818 2.3157284 1.5905145 2.1475506 1.5292094]
  [2.1560936 1.2234046 2.2082577 1.4830277 2.2137594 1.3547311]
  [1.6292869 0.9498924 1.9657116 0.808999  1.5775065 1.2946

In [ ]:
# Reading einsum strings:
"""
List each input and its axes: assign a letter to each dimension.
Find axes in inputs but not in output: those are summed over (contracted).
Find axes in output: those are kept.
Find axes in output but not in any input: those are new (broadcast).

Example — 'bchw,oc->bohw' (from a conv-like operation):
Input 1: b=batch, c=in_channels, h=height, w=width
Input 2: o=out_channels, c=in_channels
Output:  b=batch, o=out_channels, h=height, w=width

Contracted: c (in_channels) — summed over
Kept: b, h, w from input 1; o from input 2
Result: for each (batch, output_channel, h, w), sum over input_channels
→ This is a 1×1 convolution / channel mixing
"""

In [131]:
# np.tensordot() - batch operations with axis specification
# np.tensordot(a, b, axis) contracts specified axis.
a = np.random.rand(3, 4, 5).astype(np.float32)
b = np.random.rand(4, 5, 6).astype(np.float32)
result = np.tensordot(a, b, axes = ([1, 2], [0, 1]))
print(result)
print(result.shape)

print('-' * 50)

print(np.einsum('ijk,jkl->il', a, b))
print(np.einsum('ijk,jkl->il', a, b).shape)

[[6.2928605 7.3048577 5.405968  5.5817475 6.8330193 4.6662784]
 [4.9566097 5.5247107 4.9532814 4.0727296 6.058789  4.126641 ]
 [5.351355  5.5011897 5.0957737 4.6019115 5.590365  4.429463 ]]
(3, 6)
--------------------------------------------------
[[6.29286   7.3048577 5.405968  5.5817475 6.8330193 4.666278 ]
 [4.95661   5.5247107 4.9532814 4.0727296 6.0587893 4.126641 ]
 [5.351355  5.5011897 5.0957737 4.6019115 5.590365  4.429463 ]]
(3, 6)


In [136]:
# Performance
import numpy as np
import time

batch, m, n, k = 32, 128, 128, 128
a = np.random.rand(batch, m, n).astype(np.float32)
b = np.random.rand(batch, n, k).astype(np.float32)

def bench(label, fn, reps = 200):
    t0 = time.perf_counter()
    for _ in range(reps):
        fn()
    t1 = time.perf_counter()
    print(f'{label:<35} {(t1 - t0) / reps * 1000:.3f} ms/call')

print(bench('@ operator: ', lambda: a @ b))
print(bench('np.matmul: ', lambda: np.matmul(a, b)))
print(bench("np.einsum (bij,bjk->bik)", lambda: np.einsum('bij,bjk->bik', a, b)))
print(bench("np.einsum (optimize=True)", lambda: np.einsum('bij,bjk->bik', a, b, optimize = True)))

@ operator:                         2.905 ms/call
None
np.matmul:                          2.877 ms/call
None
np.einsum (bij,bjk->bik)            9.724 ms/call
None
np.einsum (optimize=True)           9.733 ms/call
None


In [ ]:
# Common Patterns
# Dot product of two vectors
np.dot(a, b)                        # or: (a * b).sum()

# Matrix × vector
A @ v                               # or: np.einsum('ij,j->i', A, v)

# Matrix × matrix
A @ B                               # or: np.einsum('ij,jk->ik', A, B)

# Batched matrix × matrix
A @ B                               # A: (..., m, n), B: (..., n, k)

# Outer product
np.outer(a, b)                      # or: a[:, None] * b[None, :]
                                    # or: np.einsum('i,j->ij', a, b)

# Row-wise dot products (batch of vectors, pairwise)
(A * B).sum(axis=1)                 # or: np.einsum('ij,ij->i', A, B)

# Pairwise distances (Euclidean)
# ||a - b||² = ||a||² + ||b||² - 2 a·b
sq_dists = (np.linalg.norm(A, axis=1)**2)[:, None] \
         + (np.linalg.norm(B, axis=1)**2)[None, :] \
         - 2 * A @ B.T             # shape (n_A, n_B)

# Cosine similarity matrix
A_norm = A / np.linalg.norm(A, axis=1, keepdims=True)
B_norm = B / np.linalg.norm(B, axis=1, keepdims=True)
cosine_sim = A_norm @ B_norm.T     # shape (n_A, n_B)

# Trace
np.einsum('ii->', A)               # or: np.trace(A)

# Diagonal
np.einsum('ii->i', A)              # or: np.diag(A)

In [13]:
# Exercise Questions
import numpy as np

# 1. Spot the bug: # Batch of 4 weight matrices (4, 3, 3) applied to batch of vectors (4, 3)
W = np.random.rand(4, 3, 3).astype(np.float32)
x = np.random.rand(4, 3).astype(np.float32)

# Intended: for each sample i, compute W[i] @ x[i]
# Which of these is correct?
# result_a = np.dot(W, x)                        # shape 
result_b = W @ x[:, :, None]                     # shape?
result_c = np.einsum('bij,bj->bi', W, x)         # shape?

# print(result_a.shape)
print(result_b.shape)
print(result_c.shape)
# Which gives (4, 3)?

print('-' * 60)

# 2. Write the einsum
a = np.random.rand(5, 3).astype(np.float32)
b = np.random.rand(3, 4).astype(np.float32)

# Write the einsum string for:
# (a) Standard matrix multiply: shape (5, 4)
print(np.einsum('ij,jk->ik', a, b).shape)
# (b) Row-wise L2 norm squared of A: shape (5,)
print(np.einsum('ij,ij->i', a, a).shape)
# (c) Outer product of first column of A and first row of B: shape (5, 4)
print(np.einsum('i,j->ij', a[:, 0], b[0, :]).shape)

print('-' * 60)

# 3. Attention Scores
np.random.seed(42)
batch, seq, d_k, d_v = 2, 6, 8, 8
Q = np.random.rand(batch, seq, d_k).astype(np.float32)
K = np.random.rand(batch, seq, d_k).astype(np.float32)
V = np.random.rand(batch, seq, d_v).astype(np.float32)

scores = np.einsum('bqk, bsk -> bqs', Q, K) / np.sqrt(d_k)   # — shape (batch, seq, seq)
print(scores.shape)
exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
output = np.einsum('bqs, bsv -> bqv', weights, V)   # — shape (batch, seq, d_v)
print(output.shape)

print('-' * 60)

# 4. Gram Matrix
# The Gram matrix G = XᵀX is used in neural style transfer and kernel SVMs
# Given X of shape (n_samples, n_features):
x = np.random.rand(10, 5).astype(np.float32)

# Compute G using:
# (a) @ operator
gram_matrix1 = x.T @ x
print(gram_matrix1.shape)
# (b) np.einsum
gram_matrix2 = np.einsum('ij,ik->jk', x, x)
print(gram_matrix2.shape)
# Verify they match with np.allclose
match = np.allclose(gram_matrix1, gram_matrix2)
print("Do both matrices match perfectly?", match)

(4, 3, 1)
(4, 3)
------------------------------------------------------------
(5, 4)
(5,)
(5, 4)
------------------------------------------------------------
(2, 6, 6)
(2, 6, 8)
------------------------------------------------------------
(5, 5)
(5, 5)
Do both matrices match perfectly? True


In [11]:
# Practice Questions
import numpy as np

# Q1. What shape does np.dot(A, B) produce when A.shape = (2, 3, 4) and B.shape = (2, 4, 5)? 
# What does A @ B produce for the same shapes? Which is correct for batched matrix multiplication?
a = np.random.rand(2, 3, 4).astype(np.float32)
b = np.random.rand(2, 4, 5).astype(np.float32)
print(np.dot(a, b))
print(np.dot(a, b).shape)
print(a @ b) # this one is correct
print((a @ b).shape)

print('\n', '=' * 50)

# Q2. Write the np.einsum string for computing the pairwise dot products between every row of 
# A (shape (m, d)) and every row of B (shape (n, d)), producing a matrix of shape (m, n).
a = np.random.rand(2, 4).astype(np.float32)
b = np.random.rand(3, 4).astype(np.float32)
print(np.einsum('ik,jk->ij', a, b))

print('\n', '=' * 50)

# Q3. You have embeddings E of shape (vocab_size, d_model) and a batch of token sequences represented 
# as indices idx of shape (batch, seq_len). Describe (in words and code) how to gather the embeddings 
# for each token without a loop, using fancy indexing.
vocab_size = 10000
d_model = 512
batch = 32
seq_len = 128

E = np.random.rand(vocab_size, d_model).astype(np.float32)
idx = np.random.randint(0, vocab_size, size = (batch, seq_len))
embeddings = E[idx]
print('E shape: ', E.shape)
print('idx shape: ', idx.shape)
print('embeddings shape: ', embeddings.shape)

print('\n', '=' * 50)

# Q4. Why is np.linalg.solve(A, b) preferred over np.linalg.inv(A) @ b? What can go wrong with the 
# inverse approach?
a = np.random.rand(5, 5).astype(np.float32)
b = np.random.rand(5).astype(np.float32)
print(np.linalg.inv(a) @ b)
print(np.linalg.solve(a, b)) # it is numerically stable

print('\n', '=' * 50)

# Q5. Interpret this einsum string: 'bnd,md->bnm'. What are the shapes of the two inputs if 
# b=2, n=10, d=64, m=20? What is the shape of the output? What operation does this compute in the 
# context of a Transformer?
b = 2
n = 10
d = 64
m = 20
# Input 1  → (2, 10, 64)
# Input 2  → (20, 64)
# Output   → (2, 10, 20)
# np.einsum('bnd,md->bnm', A, B)  is basically calculating a dot product over the d dimension.
# it computes the dot-product similarity between every query and every key.
# For every batch and every query/token, calculate its dot-product similarity with every one of the 
# m vectors. And in Transformer attention, those dot products are used to produce the attention scores 
# between queries and keys.

print('\n', '=' * 50)

# Hots Question: 'bnd,md->bnm'  = What are the two inputs, what axis is contracted, and what does the 
# output represent if this is sitting inside a cross-attention layer?
scores = np.einsum('bnd,md->bnm', Q, K)
scores = scores / np.sqrt(d)
attention_weights = softmax(scores, axis=-1)
# 'bnd,md->bnm' takes queries of shape (batch, query_len, d_model) and keys of shape (key_len, d_model). 
# It contracts/sums over the shared d_model axis, producing (batch, query_len, key_len). In 
# cross-attention, each output value represents the dot-product similarity between a query from one 
# sequence and a key from the other sequence, which becomes the basis for the cross-attention weights.

[[[[1.3327761  0.8509357  1.4222119  1.1924763  1.8120828 ]
   [1.0765212  0.8032295  0.3375015  0.7319051  1.5299627 ]]

  [[1.2977107  0.9435214  1.3768406  1.0388072  1.6781467 ]
   [1.2395189  1.3065171  0.6895094  0.6406229  1.6174715 ]]

  [[0.9871807  0.622963   1.2186344  1.0652436  1.3160008 ]
   [0.970299   0.78718287 0.2210684  0.637956   1.1868157 ]]]


 [[[1.7254456  1.1737741  1.8340938  1.5186769  2.168675  ]
   [1.6925255  1.6083665  0.691162   0.9505452  2.032052  ]]

  [[1.4346895  1.0044785  1.2076745  0.9743341  1.6694309 ]
   [1.3541064  1.3748264  0.71436393 0.6669446  1.5638673 ]]

  [[1.0514147  0.8440402  1.459935   0.97842366 1.5736699 ]
   [1.0298678  1.2319262  0.7193159  0.52677    1.607389  ]]]]
(2, 3, 2, 5)
[[[1.3327761  0.85093576 1.4222118  1.1924764  1.8120828 ]
  [1.2977105  0.9435214  1.3768407  1.0388072  1.6781467 ]
  [0.9871807  0.622963   1.2186345  1.0652436  1.3160008 ]]

 [[1.6925254  1.6083665  0.6911621  0.95054525 2.0320523 ]
  [1.3541063  

#### Pandas - Data Loading, Cleaning and Transformations

In [ ]:
# pd.read_csv() - Loading Data without running out of memory
import numpy as np
import pandas as pd

df = pd.read_csv('data.csv') # loads everything, infers dtypes (slow and memory hungry)

df = pd.read_csv('data.csv', usecols = ['user_id', 'age', 'score', 'category'], parse_dates = ['signup_dates'], 
                 dtype = {'user_id' : np.int32, 'age' : np.int8, 'score' : np.float32, 'category' : 'category'}, 
                na_values = ['NA', 'N/A', '-', ''], low_memory = False)

# chunksize = Processing Files that don't fit in RAM
chunk_results = []
for chunk in pd.read_csv('huge_file.csv', chunk_size = 100_000):
    chunk = chunk.dropna(subset = ['label'])
    chunk['feature'] = chunk['raw'].str.lower().str.strip()
    chunk_results.append(chunk[['feature', 'label']])
df = pd.concat(chunk_results, ignore_index = True)

In [ ]:
# Memory Efficient Dtypes = For optimization
df = pd.read_csv('data.csv')
print(df.dtypes)
print(df.memory_usage(deep = True).sum() / 1e6, 'MB')

print('--' * 30)

# Dtype conversion
# numeric downcasting: int64 -> int32 -> int8
df['age'] = df['age'].astype(np.int8) # -127 to 128 -- sufficient for age
df['user_id'] = df['user_id'].astype(np.int32)

# float64 -> float32
float_cols = df.select_dtypes(include = 'float 64').columns
df[float_cols] = df[float_cols].astype(np.float32)

# string/object -> category (for low-cardinality strings)
df['country'] = df['country'].astype('category')

# Automated_downcasting
def reduce_memory(df):
    for col in df.columns:
        col_type = df.col.dtype

        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].man()

            if str(col_type)[:3] == 'int':
                if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)

            elif str[col_type][:5] == 'float':
                df[col] = df[col].astype(np.float32)

        else:
            if df[col].nunique() / len(df) < 0.5:
                df[col] = df[col].astype('category')
    
    return df

df = reduce_memory(df)
print(df.memory_usage(deep = True).sum() / 1e6, 'MB after reduction')

print('--' * 30)

# df.memory_usage(deep = True) - Profiling before we hit OOM
# deep=True is essential for object columns — without it, object columns report only the pointer size 
# (8 bytes per row), not the actual string data
mem = df.memory_usage(deep = True)
print(mem.sort_values(ascending = False).head(10))
print(f'Total: {mem.sum() / 1e6:.1f} MB')

In [5]:
# .loc (selects label returns scalar, series, df) vs .iloc (selects int position returns scalar, 
# series, df) vs .at (selects label returns scalar only) vs .iat (selects int position returns scalar only)

import pandas as pd
import numpy as np

df = pd.DataFrame({'score' : [0.9, 0.7, 0.8], 'label' : ['cat', 'dog', 'cat']}, index = [10, 20, 30])
# .loc - by label (index value)
print(df.loc[10])
print(df.loc[10 : 20])
print(df.loc[10 : 'score'])

print('--' * 30)

# .iloc - by position (0, 1, 2, ...)
print(df.iloc[0])
print(df.iloc[0 : 2])
print(df.iloc[0, 0])

print('--' * 30)

# .at and .iat - scalar access, faster than .loc/.iloc for single values
print(df.at[10, 'score'])
print(df.iat[0, 0])

print('--' * 30)

score    0.9
label    cat
Name: 10, dtype: object
    score label
10    0.9   cat
20    0.7   dog
    score label
10    0.9   cat
20    0.7   dog
30    0.8   cat
------------------------------------------------------------
score    0.9
label    cat
Name: 10, dtype: object
    score label
10    0.9   cat
20    0.7   dog
0.9
------------------------------------------------------------
0.9
0.9
------------------------------------------------------------


In [ ]:
# Chained Indexing Bug
# WRONG — chained indexing: Pandas may return a copy from the first [] then modify that copy, not 
# the original DataFrame
df[df['score'] > 0.5]['label'] = 'high'   # SettingWithCopyWarning — silent failure!

# RIGHT — single indexing operation with .loc
df.loc[df['score'] > 0.5, 'label'] = 'high'   # modifies df in-place correctly

# Copy-on-Write (CoW) is the new default in Pandas 2.0+. Under CoW, the chained indexing bug is 
# eliminated because indexing always returns a copy and the assignment simply doesn't reach the original. 
# But the correct .loc pattern works in all versions — learn it once, use it everywhere.
df.loc[condition, column_name] = new_value
df['tier'] = 'low'
df.loc[df['score'] > 0.7, 'tier'] = 'high'   # For creating a new column based on a condition:
df['tier'] = np.where(df['score'] > 0.7, 'high', 'low')   # more cleanly with np.where / pd.cut

In [ ]:
# Handling Missing Data
df = pd.DataFrame({'age' : [25, np.nan, 35, np.nan, 45], 'income' : [50000, 60000, np.nan, 80000, 90000],
                  'clicked' : [1, 0, 1, np.nan, 0]})

# Detecting missing values
print(df.isnull()) # boolean df of missing locations
print(df.isnull().sum()) # missing count per column
print(df.isnull().mean()) # missing fraction per column (useful for threshold decisions)
print(df.isnull().any(axis = 1)) # true for rows with any missing value

print('--' * 30)

# .dropna() -- Remove rows or cols
print(df.dropna()) # drops rows with any missing values
print(df.dropna(subset = ['age', 'income'])) # drop only if these columnns are missing
print(df.dropna(thresh = 2)) # keep rows with at least 2 non-null values
print(df.dropna(axis = 1)) # drop columns with any missing value

print('--' * 30)

# .fillna() - Impute missing values
# Fill with a constant
print(df['clicked'].fillna(0, inplace = True))

# Fill with column stats
print(df['age'].fillna(df['age'].median(), inplace = True))
print(df['income'].fillna(df['income'].mean(), inplace = True))

# Forward fill and backward fill - for time series
print(df['price'].fillna(method = 'ffill')) # carry last valid value forward
print(df['price'].fillna(method = 'bfill')) # propagate next valid value backward

# Fill with a per-group statistic
print(df['income'] = df.groupby('category')['income'].transform(lambda : x: x.fillna(x.median())))

print('--' * 30)

# .interpolate() -- For ordered / time-series data
ts = pd.Series([1, 0, np.nan, np.nan, 4.0, 5.0])
ts.interpolate(method = 'linear')  # [1.0, 2.0, 3.0, 4.0, 5.0]
ts.interpolate(method = 'quadratic')  # polynomial interpolation

print('--' * 30)

# Missingness itself is a data so add a binary flag before imputing
df['income_missing'] = df['income'].isnull().astype(np.int8)
df['income'] = df['income'].fillna(df['income'].median())

In [11]:
# Exercise Questions
import numpy as np
import pandas as pd

# 1. Memory Reduction
np.random.seed(42)
n = 10_000
df = pd.DataFrame({
    'user_id' : np.random.randint(0, 1_00_000, n),  # int32
    'age' : np.random.randint(18, 80, n),  # int8
    'score' : np.random.rand(n),  # float32
    'country' : np.random.choice(['US', 'UK', 'DE', 'IN', 'FR'], n)  # category
})
print('Before: ', df.memory_usage(deep = True).sum() / 1e6, 'MB')

df = df.astype(dtype = {'user_id' : np.int32, 'age' : np.int8, 'score' : np.float32, 
                        'country' : 'category'})
print('After: ', df.memory_usage(deep = True).sum() / 1e6, 'MB')

print('--' * 30, '\n')

# 2. .loc vs .iloc trap
df = pd.DataFrame({'value' : [10, 20, 30, 40, 50]}, index = [2, 4, 6, 8, 10])
# Predict the output before running each line:
print(df.loc[2])  # value = 10, index = 2
print(df.iloc[2]) # value = 30, index = 6
print(df.loc[2 : 6]) # how many rows? --> 3
print(df.iloc[2 : 4]) # how many rows? --> 2

print('--' * 30, '\n')

# 3. Missing values pipeline
np.random.seed(1)
df = pd.DataFrame({
    'feature_a' : np.where(np.random.rand(200) < 0.1, np.nan, np.random.randn(200)),
    'feature_b' : np.where(np.random.rand(200) < 0.3, np.nan, np.random.randint(0, 5, 200).astype(float)),
    'label' : np.where(np.random.rand(200) < 0.05, np.nan, np.random.randint(0, 2, 200).astype(float)),
})

# Task:
# 1. Print missing rate per column
print(df.isnull().sum())
# 2. Drop rows where 'label' is missing
df.dropna(subset = ['label'], axis = 0, inplace = True)
# 3. Add a 'feature_b_missing' indicator column
df['feature_b_missing'] = df['feature_b'].isnull().astype(int)
# 4. Impute feature_a with median, feature_b with 0
df['feature_a'] = df['feature_a'].fillna(df['feature_a'].median())
df['feature_b'] = df['feature_b'].fillna(0)
# 5. Print final shape and missing count
print(df.shape)
print(df.isna().sum())

print('--' * 30, '\n')

# 4. Chained indexing bug
df = pd.DataFrame({
    'score' : [0.3, 0.8, 0.6, 0.9, 0.4], 'label' : ['a', 'b', 'a', 'b', 'a']
})

# # This silently fails — fix it:
# df[df['score'] > 0.5]['label'] = 'high'
# print(df['label'])

# Correct Version is:
df.loc[df['score'] > 0.5, 'label'] = 'high'
print(df)

print('--' * 30, '\n')

Before:  0.750132 MB
After:  0.100599 MB
------------------------------------------------------------ 

value    10
Name: 2, dtype: int64
value    30
Name: 6, dtype: int64
   value
2     10
4     20
6     30
   value
6     30
8     40
------------------------------------------------------------ 

feature_a    27
feature_b    59
label         9
dtype: int64
(191, 4)
feature_a            0
feature_b            0
label                0
feature_b_missing    0
dtype: int64
------------------------------------------------------------ 

   score label
0    0.3     a
1    0.8  high
2    0.6  high
3    0.9  high
4    0.4     a
------------------------------------------------------------ 



In [ ]:
# Practice Questions: 

# Q1. You load a CSV with pd.read_csv('data.csv'). The user_id column contains integers up to 50,000. 
# The rating column contains values between 1.0 and 5.0. The country column has 180 unique string values 
# out of 10 million rows. What dtype should each be, and why?
"""user_id : int32,  rating : float 32,  country : category"""

print('--' * 30, '\n')

# Q2. What is the difference between df.loc[0:5] and df.iloc[0:5] when the DataFrame has a default 
# integer index [0, 1, 2, ..., N]? When do they give different results?
""".loc takes labels and here the stop value is inclusive and .iloc takes indices and stop value is 
exclusive. They give different results  when index does not start from 0"""

print('--' * 30, '\n')

# Q3. You have a feature income with 15% missing values. Your model is LightGBM. Should you impute? 
# If yes, with what? If no, why not?
"""No, we will simply leave the missing values as NaN (or None / null) in your dataframe. LightGBM will 
automatically allocate them to the optimal side of the split during training."""

print('--' * 30, '\n')


# Q4. Explain why this code is wrong and produces a SettingWithCopyWarning:
# high_scorers = df[df['score'] > 0.8]
# high_scorers['tier'] = 'premium'
"""The code is wrong because it performs chained indexing, which makes it ambiguous whether you are
modifying a copy or a view of the original DataFrame.  When you attempt to add the 'tier' column in the
next line, pandas triggers the SettingWithCopyWarning to warn you that your modifications might not be 
saving correctly or could be unintentionally altering the original df."""

print('--' * 30, '\n')

# Q5. You're building a training pipeline. You compute fill_value = df_train['age'].median() and then 
# apply df_train['age'].fillna(fill_value) and df_test['age'].fillna(fill_value). Why is this correct? 
# What would go wrong if you computed the median on the full dataset before splitting?
"""It is correct becuse if we compute mediaan on full dataset it will lead to data leakage that means
the computation will learn about the testing data which should have been unseen during training and
may lead to overfitting model"""

#### Pandas Groupby, Merge, Reshape - Feature Engineering Essentials

In [ ]:
# groupby() - Four methods
